# Generación de un grafo básico con WIKIDATA

In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import requests
import time

# Configuración del endpoint de Wikidata
WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"


class WikidataQuery:
    def __init__(self):
        self.sparql = SPARQLWrapper(WIKIDATA_ENDPOINT)
        self.sparql.setReturnFormat(JSON)
        
        # User-Agent requerido por Wikidata
        self.sparql.addCustomHttpHeader("User-Agent", 
                                       "SBC-Course/1.0 (educational-use)")
    
    def execute_query(self, query):
        """Ejecutar consulta SPARQL con manejo de errores"""
        try:
            self.sparql.setQuery(query)
            results = self.sparql.query().convert()
            return results["results"]["bindings"]
        except Exception as e:
            print(f"Error en la consulta: {e}")
            return []
    
    def results_to_dataframe(self, results):
        """Convertir resultados SPARQL a DataFrame de pandas"""
        if not results:
            return pd.DataFrame()
        
        # Extraer nombres de columnas
        columns = results[0].keys()
        
        # Convertir a diccionario
        data = []
        for result in results:
            row = {}
            for col in columns:
                if col in result:
                    row[col] = result[col]["value"]
                else:
                    row[col] = None
            data.append(row)
        
        return pd.DataFrame(data)

# Instanciar la clase
wd = WikidataQuery()

In [2]:
rm_query = """
# Consulta para obtener jugadores actuales del Real Madrid CF
SELECT DISTINCT ?jugador ?declaracionEquipo ?jugadorLabel ?fechaInicio WHERE {
  
  # El jugador debe ser miembro actual del Real Madrid CF (Q8682)
  ?jugador p:P54 ?declaracionEquipo .
  ?declaracionEquipo ps:P54 wd:Q8682 .
  
  # Verificar que sea actual (sin fecha de fin o fecha de fin futura)
  ?declaracionEquipo pq:P580 ?fechaInicio .
  OPTIONAL {?declaracionEquipo pq:P582 ?fechaFin . }
  FILTER (!BOUND(?fechaFin))
  
  # El jugador debe ser futbolista
  ?jugador wdt:P106 wd:Q937857 .
  
  # Obtener etiquetas en español
  SERVICE wikibase:label { 
    bd:serviceParam wikibase:language "es,en" . 
  }
}
ORDER BY ?jugadorLabel
"""

In [3]:
rm_results = wd.execute_query(rm_query)
rm_results

[{'jugador': {'type': 'uri',
   'value': 'http://www.wikidata.org/entity/Q27342447'},
  'declaracionEquipo': {'type': 'uri',
   'value': 'http://www.wikidata.org/entity/statement/Q27342447-6c08691a-481f-8737-2bd2-f17cb94bf54d'},
  'jugadorLabel': {'xml:lang': 'es',
   'type': 'literal',
   'value': 'Andriy Lunin'},
  'fechaInicio': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime',
   'type': 'literal',
   'value': '2018-06-19T00:00:00Z'}},
 {'jugador': {'type': 'uri',
   'value': 'http://www.wikidata.org/entity/Q108159340'},
  'declaracionEquipo': {'type': 'uri',
   'value': 'http://www.wikidata.org/entity/statement/Q108159340-9f95afed-4b16-ab09-8c94-67c0f133674c'},
  'jugadorLabel': {'xml:lang': 'es', 'type': 'literal', 'value': 'Arda Güler'},
  'fechaInicio': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime',
   'type': 'literal',
   'value': '2023-07-01T00:00:00Z'}},
 {'jugador': {'type': 'uri',
   'value': 'http://www.wikidata.org/entity/Q26973015'},
  'declaracionEq

In [4]:
# Si preferimos ver los datos en un DataFrame
rm_df = wd.results_to_dataframe(rm_results)
display(rm_df.head(5))

,jugador,declaracionEquipo,jugadorLabel,fechaInicio
0,http://www.wikidata.org/entity/Q27342447,http://www.wikidata.org/entity/statement/Q2734...,Andriy Lunin,2018-06-19T00:00:00Z
1,http://www.wikidata.org/entity/Q108159340,http://www.wikidata.org/entity/statement/Q1081...,Arda Güler,2023-07-01T00:00:00Z
2,http://www.wikidata.org/entity/Q26973015,http://www.wikidata.org/entity/statement/Q2697...,Brahim Abdelkader Díaz,2021-01-01T00:00:00Z
3,http://www.wikidata.org/entity/Q26973015,http://www.wikidata.org/entity/statement/Q2697...,Brahim Abdelkader Díaz,2019-01-01T00:00:00Z
4,http://www.wikidata.org/entity/Q182308,http://www.wikidata.org/entity/statement/Q1823...,Celso Ortiz,2016-01-01T00:00:00Z


In [5]:
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL, XSD

# Crear grafo RDF para la ontología
graph = Graph()

# Definir namespaces
WD = Namespace("http://www.wikidata.org/entity/")
WDT = Namespace("http://www.wikidata.org/prop/direct/")
ONTO = Namespace("http://www.fdi.ucm.es/mia/sbc/ontology/")

# Vincular namespaces al grafo
graph.bind("wd", WD)
graph.bind("wdt", WDT)
graph.bind("onto", ONTO)
graph.bind("owl", OWL)
graph.bind("rdfs", RDFS)
graph.bind("XSD", XSD)

graph.add((ONTO[""], RDF.type, OWL.Ontology))
graph.add((ONTO[""], RDFS.label, Literal("Ontología generada desde Wikidata")))

<Graph identifier=Ndca0c1c954e247719fe7e848c0cab6ea (<class 'rdflib.graph.Graph'>)>

In [6]:
graph.add((ONTO["RMplayer"], RDF.type, OWL.Class))
graph.add((ONTO["RMplayer"], RDFS.label, Literal("Real Madrid Player")))
graph.add((ONTO["RMplayer"], RDFS.subClassOf, WDT['Q5']))


<Graph identifier=Ndca0c1c954e247719fe7e848c0cab6ea (<class 'rdflib.graph.Graph'>)>

In [7]:
for entity in rm_results:
            # Extraer URI y datos
            entity_uri = URIRef(entity["jugador"]["value"])
            entity_label = entity["jugadorLabel"]["value"]
            
            # Añadir entidad como clase OWL
            graph.add((entity_uri, RDF.type, ONTO["RMplayer"]))
            graph.add((entity_uri, RDFS.label, Literal(entity_label)))
            
            # Añadir descripción si existe
            if "description" in entity and entity["description"]:
                description = entity["description"]["value"]
                graph.add((entity_uri, RDFS.comment, Literal(description)))

In [8]:
# Guardar la ontología en un archivo Turtle
import sbc_tools as sbc
sbc.save(graph, "real_madrid_ontology.ttl")

Ontología guardada en: data\real_madrid_ontology.ttl


## Ahora vamos a añadir la antigüedad

In [9]:
graph.add((ONTO.antiguedad, RDF.type, OWL.DatatypeProperty))

<Graph identifier=Ndca0c1c954e247719fe7e848c0cab6ea (<class 'rdflib.graph.Graph'>)>

In [10]:
from datetime import datetime

for entity in rm_results:
            # Extraer URI y datos
            entity_uri = URIRef(entity["jugador"]["value"])
            entity_label = entity["jugadorLabel"]["value"]
            
            fi = entity['fechaInicio']['value']
            fecha_inicio = datetime.fromisoformat(fi.replace('Z', '+00:00'))
            fecha_actual = datetime.now()
            antiguedad = (fecha_actual.date() - fecha_inicio.date()).days // 365
            
            graph.add((entity_uri, ONTO.antiguedad, Literal(antiguedad, datatype=XSD.integer)))

In [11]:
sbc.save(graph, "real_madrid_ontology_ext.ttl")

Ontología guardada en: data\real_madrid_ontology_ext.ttl


In [12]:
query = """

SELECT ?jugador ?nombre ?antiguedad
WHERE {
    ?jugador a onto:RMplayer .
    ?jugador rdfs:label ?nombre .
    ?jugador onto:antiguedad ?antiguedad .
    FILTER(?antiguedad > 5)
}
ORDER BY DESC(?antiguedad)
"""

# Ejecutar la consulta
results = graph.query(query)


for row in results:
    jugador_id = str(row.jugador)  # Extraer solo el ID
    nombre = str(row.nombre)
    antiguedad = int(row.antiguedad)
    print(f"{nombre}: {antiguedad} años (ID: {jugador_id})")

Renato Ibarra: 14 años (ID: http://www.wikidata.org/entity/Q654903)
Daniel Carvajal: 12 años (ID: http://www.wikidata.org/entity/Q127452)
Omar Mascarell: 12 años (ID: http://www.wikidata.org/entity/Q7089760)
Javi Muñoz Jiménez: 11 años (ID: http://www.wikidata.org/entity/Q18654709)
Grégory Thil: 10 años (ID: http://www.wikidata.org/entity/Q2759198)
Matías Alustiza: 10 años (ID: http://www.wikidata.org/entity/Q592084)
Celso Ortiz: 9 años (ID: http://www.wikidata.org/entity/Q182308)
Felipe Flores Chandía: 9 años (ID: http://www.wikidata.org/entity/Q2067907)
Wilker Ángel: 9 años (ID: http://www.wikidata.org/entity/Q9096005)
Gerardo Flores Zúñiga: 8 años (ID: http://www.wikidata.org/entity/Q5550269)
Andriy Lunin: 7 años (ID: http://www.wikidata.org/entity/Q27342447)
Federico Valverde: 7 años (ID: http://www.wikidata.org/entity/Q19509544)
Javier Sánchez de Felipe: 7 años (ID: http://www.wikidata.org/entity/Q58756067)
Vinícius Júnior: 7 años (ID: http://www.wikidata.org/entity/Q28973866)
Bra

## Vamos a visualizar la ontología

Utilizamos las herramientas del archivo auxiliar sbc_tools.py

In [13]:
sbc.print_class_hierarchy(graph)


JERARQUÍA DE CLASES DE LA ONTOLOGÍA
📁 Real Madrid Player (RMplayer)
  └─ Q27342447: Andriy Lunin
  └─ Q108159340: Arda Güler
  └─ Q26973015: Brahim Abdelkader Díaz
  └─ Q182308: Celso Ortiz
  └─ Q127452: Daniel Carvajal
  └─ Q31981: David Alaba
  └─ Q112184628: Dean Huijsen
  └─ Q131416396: Diego Aguado
  └─ Q110644260: Endrick Felipe
  └─ Q19509544: Federico Valverde
  └─ Q2067907: Felipe Flores Chandía
  └─ Q23884186: Fernando Medina
  └─ Q123289726: Franco Mastantuono
  └─ Q5550269: Gerardo Flores Zúñiga
  └─ Q2759198: Grégory Thil
  └─ Q6139450: James McKinstry
  └─ Q18654709: Javi Muñoz Jiménez
  └─ Q58756067: Javier Sánchez de Felipe
  └─ Q66241169: Jude Bellingham
  └─ Q21621995: Kylian Mbappé
  └─ Q592084: Matías Alustiza
  └─ Q157584: Nelson Haedo Valdez
  └─ Q7089760: Omar Mascarell
  └─ Q654903: Renato Ibarra
  └─ Q42728914: Rodrygo Goes
  └─ Q27569376: Trent Alexander-Arnold
  └─ Q28973866: Vinícius Júnior
  └─ Q9096005: Wilker Ángel
  └─ Q86322500: jose martinez lopez
  └

In [14]:

sbc.show_graph(graph, "real_madrid_ontology.html")

out/real_madrid_ontology.html


---
# Ejercicio

## Extrae de wikidata un grafo de conocimiento del dominio que prefieras

## Nombres:
- Pedro Antonio Martín Peláez
- Álvaro Ray Martínez

# Dominio: Series de anime


In [23]:
anime_query = """
SELECT DISTINCT ?anime ?animeLabel (GROUP_CONCAT(DISTINCT ?generoLabel; separator=", ") AS ?generos) ?numEpisodios ?numTemporadas ?year 

WHERE {
  ?anime wdt:P31 wd:Q63952888 .  # Serie de animación japonesa
  ?anime rdfs:label ?animeLabel .
  OPTIONAL { ?anime wdt:P136 ?genero .
            ?genero rdfs:label ?generoLabel
            FILTER(lang(?generoLabel) = 'en')}
  OPTIONAL { ?anime wdt:P1113 ?numEpisodios . }
  OPTIONAL { ?anime wdt:P2437 ?numTemporadas . }
  OPTIONAL { ?anime wdt:P580 ?fechaInicio . 
            BIND(YEAR(?fechaInicio) as ?year) }
  FILTER(lang(?animeLabel) = 'en')
}
GROUP BY ?anime ?animeLabel ?numEpisodios ?numTemporadas ?year 
ORDER BY ?animeLabel

"""

wd = WikidataQuery()
anime_results = wd.execute_query(anime_query)
df = wd.results_to_dataframe(anime_results)

print(f"{len(df)} animes")
print(df.head())

graph = Graph()

WD = Namespace("http://www.wikidata.org/entity/")
ONTO = Namespace("http://www.fdi.ucm.es/mia/sbc/animeontology/")

graph.bind("wd", WD)
graph.bind("onto", ONTO)
graph.bind("owl", OWL)
graph.bind("rdfs", RDFS)
graph.bind("xsd", XSD)

graph.add((ONTO["AnimeOntology"], RDF.type, OWL.Ontology))
graph.add((ONTO["AnimeOntology"], RDFS.label, Literal("Ontología de series de animación japonesas (Wikidata)")))

graph.add((ONTO["AnimeSeries"], RDF.type, OWL.Class))
graph.add((ONTO["AnimeSeries"], RDFS.label, Literal("Anime Series")))

graph.add((ONTO["Genre"], RDF.type, OWL.Class))
graph.add((ONTO["Genre"], RDFS.label, Literal("Genre")))

graph.add((ONTO["hasGenre"], RDF.type, OWL.ObjectProperty))
graph.add((ONTO["hasGenre"], RDFS.domain, ONTO["AnimeSeries"]))
graph.add((ONTO["hasGenre"], RDFS.range, ONTO["Genre"]))

graph.add((ONTO["numEpisodes"], RDF.type, OWL.DatatypeProperty))
graph.add((ONTO["numEpisodes"], RDFS.domain, ONTO["AnimeSeries"]))
graph.add((ONTO["numEpisodes"], RDFS.range, XSD.integer))

graph.add((ONTO["numSeasons"], RDF.type, OWL.DatatypeProperty))
graph.add((ONTO["numSeasons"], RDFS.domain, ONTO["AnimeSeries"]))
graph.add((ONTO["numSeasons"], RDFS.range, XSD.integer))

graph.add((ONTO["startDate"], RDF.type, OWL.DatatypeProperty))
graph.add((ONTO["startDate"], RDFS.domain, ONTO["AnimeSeries"]))
graph.add((ONTO["startDate"], RDFS.range, XSD.year))

for entity in anime_results:
    anime_uri = URIRef(entity["anime"]["value"])
    anime_label = entity["animeLabel"]["value"]
    
    graph.add((anime_uri, RDF.type, ONTO["AnimeSeries"]))
    graph.add((anime_uri, RDFS.label, Literal(anime_label)))

    if "generos" in entity and entity["generos"]["value"]:
        genres = entity["generos"]["value"].split(", ")
        for genre_label in genres:
            genre_uri = URIRef(ONTO[genre_label.replace(" ", "_")])
            graph.add((genre_uri, RDF.type, ONTO["Genre"]))
            graph.add((genre_uri, RDFS.label, Literal(genre_label)))
            graph.add((anime_uri, ONTO["hasGenre"], genre_uri)) 
    
    if "numEpisodios" in entity and entity["numEpisodios"]["value"]:
        graph.add((anime_uri, ONTO["numEpisodes"], Literal(int(entity["numEpisodios"]["value"]), datatype=XSD.integer)))
    
    if "numTemporadas" in entity and entity["numTemporadas"]["value"]:
        graph.add((anime_uri, ONTO["numSeasons"], Literal(int(entity["numTemporadas"]["value"]), datatype=XSD.integer)))

    if "fechaInicio" in entity and entity["fechaInicio"]["value"]:
        graph.add((anime_uri, ONTO["startDate"], Literal(entity["fechaInicio"]["value"], datatype=XSD.dateTime)))

sbc.save(graph, "anime_ontology_ext.ttl")

4122 animes
                                       anime  \
0  http://www.wikidata.org/entity/Q107704661   
1  http://www.wikidata.org/entity/Q133282714   
2  http://www.wikidata.org/entity/Q135975942   
3  http://www.wikidata.org/entity/Q136332630   
4  http://www.wikidata.org/entity/Q119980257   

                          animeLabel  \
0                  "Deji" Meets Girl   
1                       #Compass 2.0   
2           #Kanagawa ni Sunderu Elf   
3              #Zombie Sagashitemasu   
4  'Tis Time for "Torture," Princess   

                                           generos numEpisodios numTemporadas  \
0                          fantasy anime and manga           12             1   
1                                                            12             1   
2  fantasy anime and manga, comedy anime and manga         None          None   
3                           comedy anime and manga         None          None   
4  fantasy anime and manga, comedy anime and manga   

In [25]:
sbc.print_class_hierarchy(graph)


JERARQUÍA DE CLASES DE LA ONTOLOGÍA
📁 Anime Series (AnimeSeries) [hasGenre]
  └─ Q107704661: "Deji" Meets Girl
  └─ Q133282714: #Compass 2.0
  └─ Q135975942: #Kanagawa ni Sunderu Elf
  └─ Q136332630: #Zombie Sagashitemasu
  └─ Q119980257: 'Tis Time for "Torture," Princess
  └─ Q123487890: .hack//Legend of the Twilight
  └─ Q1745049: .hack//Roots
  └─ Q932962: .hack//Sign
  └─ Q126366149: 0-Saiji Start Dash Monogatari
  └─ Q135207773: 0-sen Hayato
  └─ Q118744359: 001/7 Tom Thumb
  └─ Q136099096: 009-1
  └─ Q104218124: 07-Ghost
  └─ Q135277857: 100 Sleeping Princes and the Kingdom of Dreams
  └─ Q118140488: 11eyes
  └─ Q117303887: 16bit Sensation: Another Layer
  └─ Q108535293: 180-Byō de Kimi no Mimi o Shiwase ni Dekiru ka?
  └─ Q30061532: 18if
  └─ Q100256360: 2.43: Seiin High School Boys Volleyball Team
  └─ Q115650016: 2.5 Dimensional Seduction
  └─ Q837496: 21 Emon
  └─ Q68325523: 22/7
  └─ Q135649332: 2200-Nen Neko no Kuni Nippon
  └─ Q1186775: 30-sai no Hoken Taiiku
  └─ Q700044

In [24]:
sbc.show_graph(graph, "anime_ontology.html")

out/anime_ontology.html
